# Model Training — Fraud Detection

Trains a PySpark **RandomForest classifier** (built-in, no SynapseML) to
predict the fraud flag from transaction / account / customer features.

**Target:** `had_fraud`  **Reads:** `gold_ml_features`  **Writes:** `gold_ml_model_metrics`, `Files/models/fraud_detection_rf`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when, isnan
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
df = spark.read.table('gold_ml_features')
print(f'Feature rows: {df.count():,}')

for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

df.groupBy('had_fraud').count().show()

In [ ]:
numeric_features = [
    'amount', 'log_amount', 'transaction_hour', 'is_night', 'is_international',
    'is_high_value', 'balance', 'credit_limit', 'credit_utilisation_pct',
]
cat_cols = ['transaction_type', 'merchant_category', 'channel', 'country',
            'account_type', 'age_group', 'segment', 'region', 'risk_tier']

indexed_df = df
cat_idx_cols = []
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)
    cat_idx_cols.append(idx_col)

all_features = numeric_features + cat_idx_cols
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(indexed_df).select('features', col('had_fraud').cast('double').alias('label'))
model_df = model_df.cache()
print(f'Model rows: {model_df.count():,} | features: {len(all_features)}')

In [ ]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count():,}  Test: {test_df.count():,}')

In [ ]:
rf = RandomForestClassifier(
    featuresCol='features', labelCol='label',
    predictionCol='prediction', rawPredictionCol='rawPrediction', probabilityCol='probability',
    numTrees=80, maxDepth=10, seed=42,
)
model = rf.fit(train_df)
print('RandomForest classifier trained')

In [ ]:
predictions = model.transform(test_df)
auc = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC').evaluate(predictions)
accuracy = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy').evaluate(predictions)
f1 = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='f1').evaluate(predictions)
print(f'AUC-ROC: {auc:.4f}  Accuracy: {accuracy:.4f}  F1: {f1:.4f}')

In [ ]:
metrics = spark.createDataFrame(
    [('financial-services', 'fraud-detection', 'RandomForestClassifier',
      len(all_features), train_df.count(), test_df.count(),
      float(auc), float(accuracy), float(f1))],
    ['demo_id', 'use_case', 'model_type', 'feature_count',
     'train_rows', 'test_rows', 'auc_roc', 'accuracy', 'f1_score']
).withColumn('trained_at', current_timestamp())
metrics.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_model_metrics')
model.write().overwrite().save('Files/models/fraud_detection_rf')
model_df.unpersist()
print('Metrics + model saved')